# System Router on Colab

Thin notebook: all logic lives in the repository. Use an A100 runtime for Qwen3.5-9B unquantized; on a T4 or L4 add `--set llm.quantization=4bit` (and `llm.dtype=float16` on a T4).

Runs, checkpoints and the label cache go to Google Drive so a disconnect loses nothing; the model is downloaded to the session disk each time.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUNS = '/content/drive/MyDrive/system-router/runs'
MODEL = '/content/models/Qwen3.5-9B'

In [ ]:
!git clone --recurse-submodules https://github.com/serciex/System-Router.git /content/System-Router
%cd /content/System-Router

In [ ]:
# Chrome for MiniWoB++ (Selenium downloads a matching chromedriver itself)
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get -qq update && apt-get -qq install -y ./google-chrome-stable_current_amd64.deb
!pip -q install -r requirements.txt

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download('Qwen/Qwen3.5-9B', local_dir=MODEL)

In [ ]:
# Stage 0: unit tests and adapter conformance (no model needed)
!python -m pytest -q tests
!python scripts/check_adapters.py --adapters dom a11y "degraded:dom"

In [ ]:
# Stage 1-2: rules instead of the world model, System 1 vs System 2 alone
!python scripts/run_stage1.py --episodes 20 --route s1 --set paths.model={MODEL} --set paths.runs={RUNS}
!python scripts/run_stage1.py --episodes 20 --route s2 --set paths.model={MODEL} --set paths.runs={RUNS}

In [ ]:
# Stage 3: train the world model (resumes from RUNS/wm/latest.pt)
!python scripts/train_wm.py --set paths.model={MODEL} --set paths.runs={RUNS}